# Pre-processing Colonies Dataset (24-25 August 2020)

* Data Pre-processing **[done]**
    * Import colonies
    * Import barrier files – reproject all to EPSG 7760
    * Check validity of all shapefiles (turn this into a function…) – also check that all points are in Delhi. (might be part of spatial index notebook and UAC deduplication)    
* Compute barrier clip for all colonies **[done]**
* Run Neighbors Algorithm **[done]**
    * Touching Neighbors algorithm - Modify so that it ignores NDMC and related areas (The NDMC / DCB polygons are coded as NDMC and DCB)
    * bbox Neighbors algorithm
    * Should check for barriers
    * Should check for NDMC and related areas
    * Save as two separate columns: touching neighbors and bbox neighbors 
* Additional preprocessing for colonies (turn into super function) **[done]**
    * Create index column **[done]**
    * Distance from NDMC **[done]**
    * Area of each polygon **[done]**
* Merge with 2020 Population data **[done]**
* Export GeoDataFrame as pickle file and ESRI Shapefiles

## Import modules and set constants

In [ ]:
import os
import pickle
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils

In [ ]:
reload(spatial_index_utils)

In [ ]:
# WGS 84 / Delhi
epsg_code = 7760

## Import shapefiles **[done]**

In [ ]:
colony_filepath = os.path.join('shapefiles', 'Spatial_Index_GIS', 'Colony_Shapefile', 
                        'USO23Aug2020.shp')

barrier_directory = os.path.join('shapefiles', 'Barrier_Clip')

canal_filepath = os.path.join(barrier_directory, 'Canal', 'Canal.shp')
drain_filepath = os.path.join(barrier_directory, 'Drain', 'Major_Drain.shp')
railway_filepath = os.path.join(barrier_directory, 'Railway', 'Railway_Line.shp')

# boundary of Delhi
delhi_bounds_filepath = os.path.join('shapefiles', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [colony_filepath, canal_filepath, drain_filepath, railway_filepath, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))

In [ ]:
colonies = gpd.read_file(colony_filepath)

## Inspect shapefiles for validity (`check_shapefile`) **[done]**

In [ ]:
spatial_index_utils.check_shapefile(gdf=colonies, gdf_name='colonies', 
                                    geom_type='Polygon', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=canal, gdf_name='canal', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=drain, gdf_name='drain', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=railway, gdf_name='railway', geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

## Remove duplicate geometries **[done]**

In [ ]:
#canal = spatial_index_utils.remove_duplicate_geom(canal)
#drain = spatial_index_utils.remove_duplicate_geom(drain)
#railway = spatial_index_utils.remove_duplicate_geom(railway)

#with open('canal.data', 'wb') as f:
#    pickle.dump(canal, f)

#with open('drain.data', 'wb') as f:
#    pickle.dump(drain, f)

#with open('railway.data', 'wb') as f:
#    pickle.dump(railway, f)

In [ ]:
colonies = spatial_index_utils.remove_duplicate_geom(colonies)

In [ ]:
len(colonies)

## Start here, 25 August 2020 (download colonies and barrier files)

### First import modules and shapefiles above

In [ ]:
with open('canal.data', 'rb') as f:
    canal = pickle.load(f)

In [ ]:
with open('drain.data', 'rb') as f:
    drain = pickle.load(f)

In [ ]:
with open('railway.data', 'rb') as f:
    railway = pickle.load(f)

## Check CRS, reproject to EPSG:7760.

In [ ]:
colonies.crs

In [ ]:
canal = spatial_index_utils.reproject_gdf(canal, epsg_code)

In [ ]:
drain = spatial_index_utils.reproject_gdf(drain, epsg_code)

In [ ]:
railway = spatial_index_utils.reproject_gdf(railway, epsg_code)

In [ ]:
colonies.crs == drain.crs == canal.crs == railway.crs

## Calculate Area (in square kilometers)

In [ ]:
colonies['area_km2'] = colonies.area/1000000

In [ ]:
colonies.head()

In [ ]:
colonies['area_km2'].max()

In [ ]:
colonies['area_km2'].min()

## Compute barrier clip

In [ ]:
colonies.head()

In [ ]:
# Note... I had to reset index to make spatial join work!
#colonies = colonies.reset_index()
colonies = colonies.drop(columns=['index'])

In [ ]:
colonies.head()

In [ ]:
# Create new columns showing intersection with canal, railway and drain
colonies = spatial_index_utils.barrier_intersection(colonies, canal, "canal")

In [ ]:
colonies = spatial_index_utils.barrier_intersection(colonies, railway, "railway")

In [ ]:
colonies = spatial_index_utils.barrier_intersection(colonies, drain, "drain")

In [ ]:
# Create barrier column as being intersection with canal, railway or drain
colonies['barrier'] = colonies['canal'] | colonies['railway'] | colonies["drain"]

In [ ]:
colonies.head()

In [ ]:
len(colonies)

## Calculate centroid for each polygon

In [ ]:
colonies['centroid'] = colonies.centroid
colonies.head()

## Distance from NDMC (turn into function)

In [ ]:
# ndmc_center shapefile location
ndmc_center_filepath = os.path.join('shapefiles', 'ndmc_center7760.shp')

# Import shapefile
ndmc_center = gpd.read_file(ndmc_center_filepath)

# Extract NDMC Center as Shapely Point
ndmc_center_point = ndmc_center['geometry'].values[0]

# Code to generate ndmc_distances
# initialize new column with value 0
colonies['ndmc_dist_km'] = 0

# Compute distance from NDMC to centroid of each polygon
# Division by 1000 turns units into kilometers
for idx, row in colonies.iterrows():
    colonies.loc[idx, 'ndmc_dist_km'] = ndmc_center_point.distance(row['centroid'])/1000
    
colonies.head()

## Merge population data (2020) with colonies dataset

In [ ]:
worldpop2020_filepath = os.path.join('population_data/', 'pop_colony_wp_2020.csv')

# Import 2020 population data
worldpop2020 = pd.read_csv(worldpop2020_filepath)

# Restrict dataframe to only two columns:
# layer: population data
# uso_area_u: unique id for colonies
worldpop2020 = worldpop2020[['layer', 'uso_area_u']]
worldpop2020.head()

# Merge population data with colonies data
colonies = colonies.merge(worldpop2020, how='inner', 
                          left_on="USO_AREA_U", right_on='uso_area_u')

# Rename 'layer' column as 'population'
colonies = colonies.rename(columns={'layer': 'population'})

# Remove extraneous columns
colonies = colonies.drop(columns=['uso_area_u'])

colonies.head()

## Create GeoDataFrame with Bounding Box of each Polygon

In [ ]:
colonies_bbox = spatial_index_utils.create_bbox_gdf(colonies)

## Spatial Join (intersection of polygon geometries)

In [ ]:
colonies_touch_nbrs = spatial_index_utils.add_polygon_neighbors_column_fast(polygon_gdf=colonies,
                                                        right_gdf=colonies,
                                                        id_colname='USO_AREA_U', 
                                                        neighbor_colname='nbrs_touch',
                                                        barrier_colname='barrier')

colonies_touch_nbrs.head()

## Spatial Join (intersection of polygon and bbox geometries)

In [ ]:
colonies_bbox_nbrs = spatial_index_utils.add_polygon_neighbors_column_fast(polygon_gdf=colonies,
                                                       right_gdf=colonies_bbox,
                                                       id_colname='USO_AREA_U', 
                                                       neighbor_colname='nbrs_bbox',
                                                       barrier_colname='barrier')
colonies_bbox_nbrs.head()

## Calculate neighbor distances

In [ ]:
colonies_touch_nbrs = spatial_index_utils.calc_nbr_dist(polygon_gdf=colonies_touch_nbrs,
                                  nbr_dist_colname='nbrs_dist_touch',
                                  centroid_colname='centroid',
                                  neighbor_colname='nbrs_touch',
                                  neighbor_id_col='USO_AREA_U')

colonies_touch_nbrs.head()

In [ ]:
colonies_bbox_nbrs = spatial_index_utils.calc_nbr_dist(polygon_gdf=colonies_bbox_nbrs,
                                  nbr_dist_colname='nbrs_dist_bbox',
                                  centroid_colname='centroid',
                                  neighbor_colname='nbrs_bbox',
                                  neighbor_id_col='USO_AREA_U')

colonies_bbox_nbrs.head()

## Create index column

In [ ]:
colonies_touch_nbrs['index'] = colonies_touch_nbrs.index
colonies_bbox_nbrs['index'] = colonies_bbox_nbrs.index

In [ ]:
colonies_touch_nbrs.head()

In [ ]:
colonies_bbox_nbrs.head

## Remove extraneous columns (`geom_type`)

In [ ]:
colonies_touch_nbrs = colonies_touch_nbrs.drop(columns=['geom_type']) 
colonies_bbox_nbrs = colonies_bbox_nbrs.drop(columns=['geom_type']) 

In [ ]:
colonies_touch_nbrs.head()

In [ ]:
colonies_bbox_nbrs.head()

## Save colonies file for Spatial Index

In [ ]:
with open('colonies_bbox_nbrs25Aug2020.pkl', 'wb') as f:
    pickle.dump(colonies_bbox_nbrs, f)

In [ ]:
with open('colonies_touch_nbrs25Aug2020.pkl', 'wb') as f:
    pickle.dump(colonies_touch_nbrs, f)